In [1]:
# ============================================================
# DOMAIN 5 – Thai Image Captioning  |  v3 FAST (no training)
# Strategy: BLIP-2 (English caption) → deep_translator → Thai
#
# Why this works:
#   - BLIP-2 is SOTA pretrained image captioning, no GPU training needed
#   - Google Translate gives high-quality Thai output
#   - Runs in ~20 min on P100, no CUDA build issues
#   - Expected score: 38-45+ depending on translation quality
# ============================================================
!pip install -q deep-translator transformers accelerate Pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 2.9 MB/s eta 0:00:00


In [2]:
import os, torch, pandas as pd
from PIL import Image
from tqdm import tqdm
from transformers import BlipProcessor, BlipForConditionalGeneration
from deep_translator import GoogleTranslator
import time

TEST_DIR   = "/kaggle/input/competitions/super-ai-engineer-ss-6-thai-language-image-captioning/test/test"
SAMPLE_CSV = "/kaggle/input/competitions/super-ai-engineer-ss-6-thai-language-image-captioning/sample_submission.csv"

In [3]:
# --- Load BLIP (not BLIP-2, runs fine on P100 with no install issues) ---
# Use BLIP large for best quality; switch to base if OOM
print("Loading BLIP...")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-large")
model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-large",
    torch_dtype=torch.float32   # float32 for P100 compatibility
).to(device)
model.eval()
print("BLIP loaded.")

Loading BLIP...
Device: cuda


preprocessor_config.json:   0%|          | 0.00/445 [00:00<?, ?B/s]

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/527 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.88G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/616 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-large
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BLIP loaded.


In [4]:
# --- Inference: generate English captions then translate to Thai ---
sample_df = pd.read_csv(SAMPLE_CSV)
translator = GoogleTranslator(source='en', target='th')

predictions = []
failed_translate = 0

for idx, row in tqdm(sample_df.iterrows(), total=len(sample_df), desc="Captioning"):
    image_id = f"{int(row['image_id']):05d}"
    img_path = os.path.join(TEST_DIR, f"{image_id}.jpg")

    if not os.path.exists(img_path):
        predictions.append("ภาพนี้ไม่พบ")
        continue

    # Step 1: Generate English caption with BLIP
    image = Image.open(img_path).convert("RGB")
    inputs = processor(image, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=50,
            num_beams=5,
            length_penalty=1.0,
        )
    english_caption = processor.decode(out[0], skip_special_tokens=True)

    # Step 2: Translate to Thai
    try:
        thai_caption = translator.translate(english_caption)
        # Small delay to avoid rate limiting
        if idx % 50 == 0:
            time.sleep(0.5)
    except Exception as e:
        # Fallback: keep English if translation fails
        thai_caption = english_caption
        failed_translate += 1

    predictions.append(thai_caption)

print(f"Done. Translation failures: {failed_translate}/{len(sample_df)}")

Captioning: 100%|██████████| 2000/2000 [21:27<00:00,  1.55it/s]

Done. Translation failures: 0/2000


In [5]:
# --- Save submission ---
sample_df['caption'] = predictions
sample_df['image_id'] = sample_df['image_id'].apply(lambda x: f"{int(x):05d}")
submission_path = "/kaggle/working/submission_v3.csv"
sample_df.to_csv(submission_path, index=False)
print(f"Saved to {submission_path}")
sample_df.head(10)

Saved to /kaggle/working/submission_v3.csv


,image_id,caption
0,01354,มีภาพระยะใกล้ของแมลงสีขาวตัวเล็กๆ อยู่บนพื้น
1,01413,มีจิ้งจกตัวหนึ่งนั่งอยู่บนกิ่งก้านของต้นไม้
2,01802,ต้นไม้ที่เติบโตตามภูเขาใกล้ทะเล
3,01243,มีลิงตัวเล็กนั่งอยู่บนพื้นข้างกำแพง
4,00693,มีตั๊กแตนสองตัวนั่งอยู่บนพื้น
5,01695,มีจานสีดำพร้อมผักนานาชนิดวางอยู่
6,01735,มีตะกร้าหวายวางอยู่บนพื้นหญ้า
7,01440,มีสัตว์ใหญ่ตัวหนึ่งนั่งอยู่ในป่า
8,00296,มีเห็ดสองตัวนั่งอยู่บนพื้นผิวสีน้ำเงิน
9,01144,มีอาคารสีขาวขลิบสีแดงและมีป้ายอยู่ด้านหน้า
